In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
# Robust src path — works whether run manually or via papermill
_src_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from config import *

bucket_ts = pd.read_csv(f'{OUT_PATH}02_fine_bucket_ts.csv')
bucket_ts['sale_date'] = pd.to_datetime(bucket_ts['sale_date'])

print(f"Loaded: {len(bucket_ts):,} rows")
print(f"Date range: {bucket_ts['sale_date'].min().strftime('%b %Y')} "
      f"to {bucket_ts['sale_date'].max().strftime('%b %Y')}")

In [ ]:
all_results = {}

for (div, portal, size), seg in bucket_ts.groupby(['Division', 'Portal', 'Size']):
    
    seg_key = f"{div}_{portal}_{size}"
    
    # All months and all buckets that exist for this segment
    all_months  = sorted(seg['sale_date'].unique())
    all_buckets = sorted(seg['bucket_min'].unique())
    
    # Create every combination of bucket × month
    idx = pd.MultiIndex.from_product(
        [all_buckets, all_months],
        names=['bucket_min', 'sale_date']
    )
    
    # Reindex with zero fill — this is the Cartesian grid
    seg_grid = (seg.set_index(['bucket_min', 'sale_date'])['qty']
                   .reindex(idx, fill_value=0)
                   .reset_index())
    
    all_results[seg_key] = {
        'grid' : seg_grid,
        'div'  : div,
        'portal': portal,
        'size' : size
    }

print(f"Cartesian grids built for {len(all_results)} segments")

# Pick a pilot segment dynamically — prefer a CABIN segment
PILOT_KEY = next(
    (k for k in sorted(all_results.keys()) if '_CABIN' in k),
    sorted(all_results.keys())[0]
)
print(f"Pilot segment: {PILOT_KEY}")

pilot = all_results[PILOT_KEY]
n_buckets = pilot['grid']['bucket_min'].nunique()
n_months  = pilot['grid']['sale_date'].nunique()
print(f"\n{PILOT_KEY} grid: {n_buckets} buckets × {n_months} months "
      f"= {len(pilot['grid'])} rows")
print(f"Missing combinations filled with 0: "
      f"{(pilot['grid']['qty'] == 0).sum()}")

In [ ]:
segment_pivots = {}

for seg_key, info in all_results.items():
    grid = info['grid']
    
    # Total qty sold in this segment per month
    monthly_total = (grid.groupby('sale_date')['qty']
                         .sum()
                         .rename('month_total'))
    
    # Join monthly total back and compute % share
    grid = grid.merge(monthly_total, on='sale_date')
    grid['pct_share'] = np.where(
        grid['month_total'] > 0,
        (grid['qty'] / grid['month_total'] * 100).round(4),
        0
    )
    
    # Pivot: rows = bucket_min, columns = month, values = pct_share
    pivot = grid.pivot_table(
        index='bucket_min',
        columns='sale_date',
        values='pct_share',
        fill_value=0
    )
    pivot = pivot.sort_index()  # ascending price order
    
    segment_pivots[seg_key] = {
        'pivot' : pivot,
        'div'   : info['div'],
        'portal': info['portal'],
        'size'  : info['size']
    }

print(f"Pivots built for {len(segment_pivots)} segments")
print(f"\nHL/CABIN pivot shape: {segment_pivots[PILOT_KEY]['pivot'].shape}")
print(f"(rows=buckets, cols=months)")
print(f"\nColumn sum check — each month should sum to ~100%:")
col_sums = segment_pivots[PILOT_KEY]['pivot'].sum(axis=0)
print(f"  Min: {col_sums.min():.1f}%  Max: {col_sums.max():.1f}%  Mean: {col_sums.mean():.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

pivot = segment_pivots[PILOT_KEY]['pivot']

# Left chart: heatmap (the Excel pivot table, visualised)
sns.heatmap(
    pivot,
    ax=axes[0],
    cmap='YlOrRd',
    cbar_kws={'label': '% share of monthly qty'},
    xticklabels=[pd.Timestamp(c).strftime('%b %y') 
                 for c in pivot.columns],
    yticklabels=pivot.index
)
axes[0].set_title(f'{PILOT_KEY} — % share heatmap\n(rows=price buckets, cols=months)',
                  fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Bucket min price (₹)')
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# Right chart: trend lines for top 10 buckets by total volume
top_buckets = pivot.sum(axis=1).nlargest(10).index
for bucket in top_buckets:
    axes[1].plot(
        range(len(pivot.columns)),
        pivot.loc[bucket],
        marker='o', markersize=3,
        label=f'₹{bucket}-{bucket+100}',
        alpha=0.8, linewidth=1.5
    )

axes[1].set_title('HL / CABIN — top 10 buckets trend lines\n(this is what analyst was reading visually)',
                  fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('% share of segment qty')
axes[1].set_xticks(range(len(pivot.columns)))
axes[1].set_xticklabels(
    [pd.Timestamp(c).strftime('%b %y') for c in pivot.columns],
    rotation=45, fontsize=7
)
axes[1].legend(fontsize=7, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_PATH}03_hl_cabin_trend_pivot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_hl_cabin_trend_pivot.png")

In [7]:
import pickle

# Save all pivots for notebook 04
with open(f'{OUT_PATH}03_segment_pivots.pkl', 'wb') as f:
    pickle.dump(segment_pivots, f)

print("PIVOT SUMMARY — ALL SEGMENTS")
print(f"{'Segment':<20} {'Buckets':>8} {'Months':>8} {'Top bucket %':>14}")
print("-" * 55)

for seg_key, info in sorted(segment_pivots.items()):
    pivot = info['pivot']
    top_bucket_pct = pivot.sum(axis=1).max() / pivot.sum(axis=1).sum() * 100
    print(f"  {info['div']}/{info['size']:<14} "
          f"{pivot.shape[0]:>8} "
          f"{pivot.shape[1]:>8} "
          f"{top_bucket_pct:>13.1f}%")

print(f"\nSaved: 03_segment_pivots.pkl")
print(f"Next: 04_clustering.ipynb")

PIVOT SUMMARY — ALL SEGMENTS
Segment               Buckets   Months   Top bucket %
-------------------------------------------------------
  BP/Single               16       24          40.4%
  BP/Single                8       19          40.9%
  BP/Single               20       24          19.3%
  BP/Single               15       24          50.8%
  BP/Single               21       24          13.7%
  BP/Single               16       24          35.3%
  BP/Single               21       24          22.1%
  BS/Single                1       20         100.0%
  BS/Single                1        2         100.0%
  BS/Single                2       10          80.0%
  BS/Single                2        8          85.8%
  BS/Single                8       24          68.0%
  BS/Single                2        7          57.1%
  BS/Single                3       23          78.1%
  DF/DF                    6       24          44.6%
  DF/DFT                   8       24          42.1%
  DF/DF      